In [3]:
import pandas as pd
import os

In [4]:
os.chdir(r"C:\dev\regression_mlops")
data=pd.read_csv(r"artifacts\train\training_set.csv")

In [5]:
data

,Unnamed: 0,Age,Sex,Job,Housing,Saving accounts,Checking account,Credit amount,Duration,Purpose,Risk
0,886,34,male,2,own,NaN,moderate,2825,24,business,good
1,488,35,male,1,rent,moderate,NaN,1418,10,car,good
2,265,37,male,2,own,little,moderate,802,15,radio/TV,bad
3,112,28,male,1,rent,little,moderate,6260,18,car,good
4,650,50,male,3,free,little,little,7476,48,education,good
...,...,...,...,...,...,...,...,...,...,...,...
795,289,48,male,2,own,little,little,1024,24,radio/TV,bad
796,109,35,male,2,own,quite rich,moderate,1410,14,business,good
797,907,27,male,2,own,NaN,moderate,3711,36,education,good
798,480,23,female,1,own,little,moderate,3573,12,radio/TV,good


In [6]:
data.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 800 entries, 0 to 799
Data columns (total 11 columns):
 #   Column            Non-Null Count  Dtype 
---  ------            --------------  ----- 
 0   Unnamed: 0        800 non-null    int64 
 1   Age               800 non-null    int64 
 2   Sex               800 non-null    object
 3   Job               800 non-null    int64 
 4   Housing           800 non-null    object
 5   Saving accounts   656 non-null    object
 6   Checking account  495 non-null    object
 7   Credit amount     800 non-null    int64 
 8   Duration          800 non-null    int64 
 9   Purpose           800 non-null    object
 10  Risk              800 non-null    object
dtypes: int64(5), object(6)
memory usage: 68.9+ KB


In [7]:
data.dtypes

Unnamed: 0           int64
Age                  int64
Sex                 object
Job                  int64
Housing             object
Saving accounts     object
Checking account    object
Credit amount        int64
Duration             int64
Purpose             object
Risk                object
dtype: object

In [8]:
from src.utils.common import read_yaml, create_directories


In [9]:
file=read_yaml("schema.yaml")

In [10]:
value=file.COLUMNS.values()
value

dict_values(['int64', 'int64', 'object', 'int64', 'object', 'object', 'object', 'int64', 'int64', 'object'])

In [11]:
key=file.COLUMNS.keys()
key

dict_keys(['Unnamed', 'Age', 'Sex', 'Job', 'Housing', 'Saving accounts', 'Checking account', 'Credit amount', 'Duration', 'Purpose'])

In [12]:
list(data.columns)

['Unnamed: 0',
 'Age',
 'Sex',
 'Job',
 'Housing',
 'Saving accounts',
 'Checking account',
 'Credit amount',
 'Duration',
 'Purpose',
 'Risk']

In [13]:
data.isna().sum()

Unnamed: 0            0
Age                   0
Sex                   0
Job                   0
Housing               0
Saving accounts     144
Checking account    305
Credit amount         0
Duration              0
Purpose               0
Risk                  0
dtype: int64

In [14]:
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
import numpy as np

In [16]:
y=data["Risk"]
x=data.drop(columns=["Risk","Unnamed: 0"])

In [17]:
strings=list(x.select_dtypes(include="object").columns)
strings

['Sex', 'Housing', 'Saving accounts', 'Checking account', 'Purpose']

In [19]:
intigers=list(x.select_dtypes(exclude="object").columns)
intigers

['Age', 'Job', 'Credit amount', 'Duration']

In [20]:
from sklearn.preprocessing import StandardScaler,OneHotEncoder

In [40]:
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier

# Build ColumnTransformer
preprocessor = ColumnTransformer([
    # Numeric: impute mean/median, then scale
    ('num', Pipeline([
        ('imputer', SimpleImputer(strategy='median')),  # or 'mean'
        ('scaler', StandardScaler())
    ]), intigers),
    
    # Categorical: impute with most frequent, then one-hot encode
    ('cat', Pipeline([
        ('imputer', SimpleImputer(strategy='most_frequent')),
        ('encoder', OneHotEncoder(handle_unknown='ignore', sparse_output=False))
    ]), strings)
])

# Create full pipeline (preprocessing + model)
pipeline = Pipeline([
    ('preprocessor', preprocessor),
    ('classifier', LogisticRegression())
])

# Fit and predict
pipeline.fit(x[:650],y[:650])   # example target
predictions = pipeline.predict(x[650:])

In [41]:
from sklearn.metrics import classification_report

print(classification_report(y[650:], predictions))

              precision    recall  f1-score   support

         bad       0.47      0.19      0.27        43
        good       0.74      0.92      0.82       107

    accuracy                           0.71       150
   macro avg       0.60      0.55      0.54       150
weighted avg       0.66      0.71      0.66       150



In [ ]:
model_path="artifacts\model\model.pkl"

In [ ]:
import os
import sys

import numpy as np 
import pandas as pd
import pickle
from sklearn.metrics import r2_score
from sklearn.model_selection import GridSearchCV



def evaluate_models(X_train, y_train,X_test,y_test,models,param):
        report = {}

        for i in range(len(list(models))):
            model = list(models.values())[i]
            para=param[list(models.keys())[i]]

            gs = GridSearchCV(model,para,cv=3)
            gs.fit(X_train,y_train)

            model.set_params(**gs.best_params_)
            model.fit(X_train,y_train)

            #model.fit(X_train, y_train)  # Train model

            y_train_pred = model.predict(X_train)

            y_test_pred = model.predict(X_test)

            train_model_score = r2_score(y_train, y_train_pred)

            test_model_score = r2_score(y_test, y_test_pred)

            report[list(models.keys())[i]] = test_model_score

        return report
    